# Project 3 — GPU Training Lab (CPU vs GPU, FP32/FP16/BF16)
Run the P2 MLP on CPU and GPU; sweep batch size; compare precisions.
**Goal:** answer *why does a GPU accelerate ML?* with numbers: samples/sec, time, memory.

In [ ]:
import os  # os for results dir
import time  # time for benchmarking
import csv  # csv for benchmark tables
import numpy as np  # numpy for data prep fallback
import matplotlib.pyplot as plt  # plots: batch-size vs throughput
import torch  # torch core
import torch.nn as nn  # layers
from torch.utils.data import TensorDataset, DataLoader  # batching
torch.manual_seed(42)  # seed for reproducibility
np.random.seed(42)  # seed numpy too
os.makedirs('./results', exist_ok=True)  # ensure results dir
HAS_CUDA = torch.cuda.is_available()  # detect GPU (False on this CPU-only box -> CPU rows only)
DEV = torch.device('cuda' if HAS_CUDA else 'cpu')  # pick device once
print('device =', DEV, '| cuda =', HAS_CUDA)  # log device for report
if HAS_CUDA:  # only query GPU props when present
    print(torch.cuda.get_device_name(0))  # GPU model string
    print(f'mem total: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')  # total VRAM

In [ ]:
def load_tensors(n_in_expected=None):  # same MNIST->digits fallback as P2
    try:  # try MNIST
        from torchvision import datasets  # optional dep
        tr = datasets.MNIST(root='/tmp/data', train=True, download=True)  # train
        te = datasets.MNIST(root='/tmp/data', train=False, download=True)  # test
        Xtr = tr.data.float().reshape(-1, 784) / 255.0  # flatten+scale
        ytr = tr.targets.long()  # labels
        Xte = te.data.float().reshape(-1, 784) / 255.0  # test X
        yte = te.targets.long()  # test y
        return Xtr, ytr, Xte, yte  # return
    except Exception as e:  # fallback
        print('fallback digits:', e)  # log
        from sklearn.datasets import load_digits  # digits
        from sklearn.model_selection import train_test_split  # split
        d = load_digits()  # load
        X = (d.data / 16.0).astype(np.float32)  # scale
        y = d.target.astype(np.int64)  # labels
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)  # split
        return torch.from_numpy(Xtr), torch.from_numpy(ytr), torch.from_numpy(Xte), torch.from_numpy(yte)  # to tensors
Xtr, ytr, Xte, yte = load_tensors()  # load once on CPU (move to device per-experiment)
print('shapes:', tuple(Xtr.shape), tuple(Xte.shape))  # confirm

In [ ]:
class MLP(nn.Module):  # same 128-hidden MLP as P2 (keeps comparison fair)
    def __init__(self, n_in, n_hid=256, n_out=10):  # slightly wider (256) to make GPU advantage visible
        super().__init__()  # parent init
        self.fc1 = nn.Linear(n_in, n_hid)  # input -> hidden
        self.relu = nn.ReLU()  # nonlinearity
        self.fc2 = nn.Linear(n_hid, n_out)  # hidden -> logits
    def forward(self, x):  # forward graph
        return self.fc2(self.relu(self.fc1(x)))  # logits out (CE applies softmax)
def train_config(device, dtype, batch, epochs=3):  # benchmark one (device, dtype, batch) point
    model = MLP(Xtr.shape[1]).to(device).to(dtype)  # build model directly in target dtype+device
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)  # Adam (lr fixed so dtype/device is the only change)
    loss_fn = nn.CrossEntropyLoss()  # standard CE loss
    ds = TensorDataset(Xtr, ytr)  # CPU-resident dataset (batches moved per step -> includes H2D cost, realistic)
    dl = DataLoader(ds, batch_size=batch, shuffle=True)  # loader for this batch size
    if device.type == 'cuda':  # warmup only matters on GPU (kernels autotune/caches)
        torch.cuda.synchronize()  # sync before timing (async GPU ops!)
    t0 = time.time()  # start timer
    for _ in range(epochs):  # short run: throughput matters, not convergence
        for xb, yb in dl:  # batch loop
            xb, yb = xb.to(device).to(dtype), yb.to(device)  # move inputs to device+dtype (labels stay long!)
            opt.zero_grad()  # clear grads
            loss = loss_fn(model(xb), yb)  # forward+loss
            loss.backward()  # backward
            opt.step()  # update
    if device.type == 'cuda':  # must sync: CUDA kernels are async, else timer lies
        torch.cuda.synchronize()  # wait for all kernels to finish
        mem = torch.cuda.max_memory_allocated() / 1e6  # peak VRAM in MB for this config
        torch.cuda.reset_peak_memory_stats()  # reset peak counter for next config
    else:  # CPU has no VRAM counter
        mem = 0.0  # record 0 for CSV consistency
    dt = time.time() - t0  # elapsed wall time
    sps = len(Xtr) * epochs / dt  # throughput = total samples / seconds
    return dt, sps, mem  # return time, throughput, memory
dt, sps, mem = train_config(torch.device('cpu'), torch.float32, 64, epochs=1)  # smoke test: 1 CPU epoch
print(f'smoke: {dt:.2f}s, {sps:.0f} samples/sec')  # confirm harness works before full sweep

In [ ]:
rows = []  # collect benchmark rows for CSV + plot
BATCHES = [32, 256, 2048]  # three batch sizes: small (latency-bound) -> large (throughput-bound)
DTYPES_CPU = [torch.float32]  # CPU: FP32 only (FP16/BF16 training needs GPU/amp; keeps table honest)
for b in BATCHES:  # sweep batch sizes on CPU
    dt, sps, mem = train_config(torch.device('cpu'), torch.float32, b, epochs=3)  # 3 epochs per point
    rows.append(('cpu', 'fp32', b, dt, sps, mem))  # record row
    print(f'cpu fp32 batch={b:5d} time={dt:6.2f}s sps={sps:8.0f}')  # live log
if HAS_CUDA:  # GPU sweep only when GPU exists (skips cleanly on CPU-only machines)
    for b in BATCHES:  # same batch sizes for direct comparison
        for name, dt_ in [('fp32', torch.float32), ('bf16', torch.bfloat16)]:  # FP32 vs BF16 (BF16 needs Ampere+; GB10 OK)
            dt, sps, mem = train_config(torch.device('cuda'), dt_, b, epochs=3)  # benchmark point
            rows.append(('cuda', name, b, dt, sps, mem))  # record
            print(f'cuda {name} batch={b:5d} time={dt:6.2f}s sps={sps:8.0f} mem={mem:.0f}MB')  # log with VRAM
else:  # explain the skip so the notebook reads well on CPU boxes
    print('No CUDA: GPU rows skipped. Run this cell on DGX Spark to fill the cuda rows.')  # guidance
with open('./results/gpu_lab.csv', 'w', newline='') as f:  # export benchmark table
    w = csv.writer(f)  # writer
    w.writerow(['device', 'dtype', 'batch', 'time_s', 'samples_per_sec', 'peak_mem_MB'])  # header
    w.writerows(rows)  # all rows
print('saved ./results/gpu_lab.csv with', len(rows), 'rows')  # confirm

In [ ]:
import csv as _csv  # local import for the plotting cell (keeps cell self-contained)
xs, ys, labels = [], [], []  # x=batch, y=throughput, labels=device+dtype
seen = {}  # group series by (device,dtype)
with open('./results/gpu_lab.csv') as f:  # read back the CSV we just wrote
    for r in list(_csv.DictReader(f)):  # parse each row as dict
        key = (r['device'], r['dtype'])  # series key
        seen.setdefault(key, ([], []))  # init series lists on first sight
        seen[key][0].append(int(r['batch']))  # x: batch size
        seen[key][1].append(float(r['samples_per_sec']))  # y: throughput
fig, ax = plt.subplots(figsize=(7, 4))  # one figure for the scaling plot
for (dev, dt_), (xb, yb) in seen.items():  # plot one line per series
    ax.plot(xb, yb, marker='o', label=f'{dev}/{dt_}')  # line+markers with legend label
ax.set_xscale('log')  # log x: batch sizes span 32..2048
ax.set(title='Throughput vs batch size (higher is better)', xlabel='batch size (log)', ylabel='samples/sec')  # labels
ax.legend()  # show legend
ax.grid(True, which='both', alpha=0.3)  # grid for readability
fig.tight_layout()  # fix layout
fig.savefig('./results/throughput.png', dpi=120)  # save for README
plt.show()  # display
print('Takeaway: GPU wins when batches are large enough to saturate SMs; small batches look CPU-competitive (launch + H2D overhead dominate).')  # interpretation

## README prompts
Why does the GPU win (thousands of parallel FPUs + HBM bandwidth)? When does it NOT (tiny batch, transfer-bound)? What did BF16 change (memory ~halved, throughput up on Tensor Cores, accuracy ~same here)?